# Phase 2 — Model Benchmarking (RQ1)

**Objective:** Train ResNet-50, EfficientNet-B3 and ViT-Base on the **pristine** APTOS split, then evaluate each on:
1. The clean test set
2. All 9 degraded variants (3 types × 3 levels)

**Outputs (under `Drive/Thesis/results/phase2_model_benchmarking/`):**
- `metrics/training_history_<model>.csv`
- `metrics/stress_test_results.csv` — long-format (model, degradation, level, accuracy, f1, auc)
- `plots/accuracy_vs_degradation_<type>.png` × 3
- `plots/training_curves_<model>.png` × 3
- Checkpoints: `Drive/Thesis/checkpoints/<model>_best.pt`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q timm

## Config

In [ ]:
from pathlib import Path

DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
PHASE_DIR       = RESULTS_ROOT / 'phase2_model_benchmarking'
PHASE1_DIR      = RESULTS_ROOT / 'phase1_data_engineering'
for sub in ('metrics', 'plots', 'samples', 'logs'):
    (PHASE_DIR / sub).mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'

APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('train_images') if p.is_dir())
PRISTINE_CSV = PHASE1_DIR / 'metrics' / 'pristine_split.csv'

NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
IMAGE_SIZE  = 224
SEED        = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15

MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50',
               'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}

TRAIN_CFG = dict(batch_size=32, num_workers=2, epochs=15,
                 lr=3e-4, weight_decay=1e-4, label_smoothing=0.05,
                 mixed_precision=True)

DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Datasets & loaders

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T

def make_transform(train=False):
    tfs = [T.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train:
        tfs += [T.RandomHorizontalFlip(), T.RandomRotation(15), T.ColorJitter(0.1, 0.1, 0.1)]
    tfs += [T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
    return T.Compose(tfs)

class APTOSDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.images_dir / f"{row['id_code']}.png").convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

class DegradedDataset(Dataset):
    def __init__(self, root, transform):
        self.root = Path(root); self.df = pd.read_csv(self.root / 'manifest.csv')
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['rel_path']).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

# Pristine train/val/test split — deterministic
full_pristine = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, make_transform(train=True))
n = len(full_pristine); n_test = int(n * TEST_SPLIT); n_val = int(n * VAL_SPLIT); n_train = n - n_val - n_test
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds, test_ds = random_split(full_pristine, [n_train, n_val, n_test], generator=g)

# Override eval transforms (no aug)
for ds in (val_ds, test_ds):
    ds.dataset = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, make_transform(train=False))

common = dict(batch_size=TRAIN_CFG['batch_size'],
              num_workers=TRAIN_CFG['num_workers'], pin_memory=True)
train_loader = DataLoader(train_ds, shuffle=True, **common)
val_loader   = DataLoader(val_ds,   shuffle=False, **common)
test_loader  = DataLoader(test_ds,  shuffle=False, **common)

# Save the test ids so Phase 3+ evaluate on the same images
test_ids = [test_ds.dataset.df.iloc[i]['id_code']
            for i in test_ds.indices]
pd.DataFrame({'id_code': test_ids}).to_csv(
    PHASE_DIR / 'metrics' / 'test_ids.csv', index=False)

print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

## Model factory

In [ ]:
import timm
import torch.nn as nn

def build_model(name, pretrained=True):
    model = timm.create_model(TIMM_NAMES[name], pretrained=pretrained, num_classes=NUM_CLASSES)
    model._dr_arch = name  # used later for XAI dispatch
    return model

## Train / evaluate helpers

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        prob = torch.softmax(model(x), 1).cpu().numpy()
        probs.append(prob); ps.append(prob.argmax(1)); ys.append(y.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); pr = np.concatenate(probs)
    out = {'accuracy': accuracy_score(y, p),
           'f1_macro': f1_score(y, p, average='macro')}
    try:
        out['auc_macro_ovr'] = roc_auc_score(y, pr, average='macro', multi_class='ovr')
    except ValueError:
        out['auc_macro_ovr'] = float('nan')
    return out

def train_model(model, ckpt_path, history_csv,
                epochs=TRAIN_CFG['epochs'], lr=TRAIN_CFG['lr'],
                wd=TRAIN_CFG['weight_decay'], ls=TRAIN_CFG['label_smoothing'],
                amp=TRAIN_CFG['mixed_precision']):
    model = model.to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    crit  = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler = GradScaler(enabled=amp)
    best_f1, history = -1.0, []
    for ep in range(epochs):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f'epoch {ep+1}/{epochs}', leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optim); scaler.update()
            running += loss.item() * x.size(0); seen += x.size(0)
            pbar.set_postfix(loss=running / seen)
        sched.step()
        m = evaluate(model, val_loader)
        history.append({'epoch': ep+1, 'train_loss': running/seen, **m})
        print(f'  val: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}  auc={m["auc_macro_ovr"]:.4f}')
        if m['f1_macro'] > best_f1:
            best_f1 = m['f1_macro']
            torch.save({'state_dict': model.state_dict(),
                        'arch': model._dr_arch, 'val': m}, ckpt_path)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    return pd.DataFrame(history)

## Train all three models

Checkpoints + per-model training history are saved to Drive.

In [ ]:
histories = {}
for name in MODEL_NAMES:
    print(f'\n========== Training {name} ==========')
    model = build_model(name, pretrained=True)
    ckpt  = CHECKPOINTS_DIR / f'{name}_best.pt'
    hist  = PHASE_DIR / 'metrics' / f'training_history_{name}.csv'
    histories[name] = train_model(model, ckpt, hist)
    del model; torch.cuda.empty_cache()

## Training curves (plot per model)

In [ ]:
import matplotlib.pyplot as plt
for name, h in histories.items():
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h['epoch'], h['train_loss'], label='train loss')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].set_title(f'{name} — train loss')
    ax[1].plot(h['epoch'], h['accuracy'], label='val acc')
    ax[1].plot(h['epoch'], h['f1_macro'], label='val F1')
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('score'); ax[1].set_title(f'{name} — val metrics')
    ax[1].legend()
    plt.tight_layout()
    out = PHASE_DIR / 'plots' / f'training_curves_{name}.png'
    plt.savefig(out, dpi=150); plt.show()
    print('Saved:', out)

## Stress test — every model on every (degradation, level)

Produces one long-format CSV that drives the comparison plots.

In [ ]:
# Build degraded test loaders, restricted to the held-out test ids
test_id_set = set(pd.read_csv(PHASE_DIR / 'metrics' / 'test_ids.csv')['id_code'])

class DegradedSubset(DegradedDataset):
    def __init__(self, root, keep_ids, transform):
        super().__init__(root, transform)
        self.df = self.df[self.df['id_code'].astype(str).isin(map(str, keep_ids))].reset_index(drop=True)

deg_loaders = {}
for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        ds = DegradedSubset(DEGRADED_DIR / k / l, test_id_set, make_transform(train=False))
        deg_loaders[(k, l)] = DataLoader(ds, shuffle=False, **common)
        print(f'  {k}/{l}: {len(ds)} test images')

rows = []
for name in MODEL_NAMES:
    print(f'\n--- Evaluating {name} ---')
    model = build_model(name, pretrained=False).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)
    model.load_state_dict(state['state_dict'])

    m = evaluate(model, test_loader)
    rows.append({'model': name, 'degradation': 'clean', 'level': 'none', **m})
    print(f'  clean: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')

    for (k, l), loader in deg_loaders.items():
        m = evaluate(model, loader)
        rows.append({'model': name, 'degradation': k, 'level': l, **m})
        print(f'  {k}/{l}: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')
    del model; torch.cuda.empty_cache()

stress_df = pd.DataFrame(rows)
stress_path = PHASE_DIR / 'metrics' / 'stress_test_results.csv'
stress_df.to_csv(stress_path, index=False)
print('\nSaved:', stress_path)
stress_df

## Accuracy-vs-degradation plots (one per degradation type)

In [ ]:
level_order = ['none', 'low', 'mid', 'high']
for kind in DEGRADATION_TYPES:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name in MODEL_NAMES:
        sub = stress_df[((stress_df['degradation'] == kind) | (stress_df['degradation'] == 'clean'))
                        & (stress_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        sub = sub.sort_values('level')
        ax.plot(sub['level'], sub['accuracy'], marker='o', label=name)
    ax.set_title(f'Accuracy vs {kind} severity')
    ax.set_xlabel('Degradation level'); ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = PHASE_DIR / 'plots' / f'accuracy_vs_degradation_{kind}.png'
    plt.savefig(out, dpi=150); plt.show()
    print('Saved:', out)

## Headline summary table

In [ ]:
pivot = stress_df.pivot_table(index=['degradation', 'level'],
                              columns='model', values='accuracy').round(3)
pivot.to_csv(PHASE_DIR / 'metrics' / 'accuracy_pivot.csv')
pivot

---
**Phase 2 complete.** Proceed to `03_phase3_xai_benchmark.ipynb`.